In [0]:

# Step 1: Define the lab folders
base_path = "/Volumes/workspace/default/employee_data/lab1"

source_path = f"{base_path}/source"
schema_path = f"{base_path}/schema"
checkpoint_path = f"{base_path}/checkpoint"

# Create the folders
dbutils.fs.mkdirs(source_path)
dbutils.fs.mkdirs(schema_path)
dbutils.fs.mkdirs(checkpoint_path)

# Verify folder creation
display(dbutils.fs.ls(base_path))


In [0]:
data = [
    (1, "Amit", "IT", 45000),
    (2, "Rahul", "HR", 35000),
    (3, "Priya", "Finance", 50000)
]

columns = ["id", "name", "department", "salary"]

df = spark.createDataFrame(data, columns)

display(df)

In [0]:
source_path = "/Volumes/workspace/default/employee_data/lab1"

df.coalesce(1).write.mode("overwrite").option("header", True).csv(source_path)

#Saves employee data as a CSV file.
#Stores it in your Unity Catalog Volume.
#Prepares the file for Auto Loader ingestion.

In [0]:
source_path = "/Volumes/workspace/default/employee_data/lab1/stream_source"
schema_path = "/Volumes/workspace/default/employee_data/lab1/stream_schema"

#Auto Loader detects CSV files in your Volume.
df_stream = (
    spark.readStream #readStream creates a streaming DataFrame.
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)#schemaLocation stores schema information.
    .option("header", "true")
    .load(source_path)
)

import time
checkpoint_path = f"/Volumes/workspace/default/employee_data/lab1/checkpoint_{int(time.time())}"

query = (
    df_stream.writeStream
    .outputMode("append")
    .trigger(availableNow=True)
    .format("memory")
    .queryName("employee_stream_view")
    .option("checkpointLocation", checkpoint_path)
    .start()
)

query.awaitTermination()
display(spark.sql("SELECT * FROM employee_stream_view"))

In [0]:
import time
bronze_path = f"/Volumes/workspace/default/employee_data/lab1/checkpoint_bronze_{int(time.time())}"

query = (
    df_stream.writeStream
    .format("delta")
    .option("checkpointLocation", bronze_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("workspace.default.bronze_employees")
)

In [0]:
%sql
SELECT * 
FROM workspace.default.bronze_employees;

In [0]:
print("Streaming active:", query.isActive) #checks whether your streaming query is running
print("Status:", query.status) #shows what the stream is currently doing.

In [0]:
query.stop()
#Stops the active streaming query.
#Helps avoid unnecessary compute usage in Free Edition.
#Keeps your checkpoint data available for recovery.
print(query.isActive)

In [0]:
# Now we add a new employee and check Auto Loader is detect or not 
new_data = [(4, "Neha", "IT", 40000)]

new_df = spark.createDataFrame(
    new_data,
    ["id", "name", "department", "salary"]
)

new_df.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv("/Volumes/workspace/default/employee_data/lab1/new_employee")

In [0]:
# Now we restart the Auto Loader
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option(
        "cloudFiles.schemaLocation",
        "/Volumes/workspace/default/employee_data/lab1/schema"
    )
    .option("header", "true")
    .load("/Volumes/workspace/default/employee_data/lab1/new_employee")
)

query = (
    df_stream.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/employee_data/lab1/new_checkpoint"
    )
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("default.bronze_new_employees")
)

In [0]:
%sql
SELECT * FROM workspace.default.bronze_employees;

In [0]:
checkpoint_path = "/Volumes/workspace/default/employee_data/lab1/new_checkpoint"

display(dbutils.fs.ls(checkpoint_path))

In [0]:
source_path = "/Volumes/workspace/default/employee_data/lab1/source"

schema_path = "/Volumes/workspace/default/employee_data/lab1/schema"

df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .load(source_path)
)

import time
checkpoint_path = f"/Volumes/workspace/default/employee_data/lab1/checkpoint_{int(time.time())}"

query = (
    df_stream.writeStream
    .outputMode("append")
    .trigger(availableNow=True)
    .format("memory")
    .queryName("employee_stream_view")
    .option("checkpointLocation", checkpoint_path)
    .start()
)

query.awaitTermination()
display(spark.sql("SELECT * FROM employee_stream_view"))

In [0]:
source_path = "/Volumes/workspace/default/employee_data/lab1/new_employee"
schema_path = "/Volumes/workspace/default/employee_data/lab1/schema"

df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .load(source_path)
)

query = (
    df_stream.writeStream
    .format("console")
    .option("checkpointLocation",
            "/Volumes/workspace/default/employee_data/lab1/microbatch_checkpoint")
    .trigger(availableNow=True)
    .start()
)

In [0]:
#Creat a streaming data
from pyspark.sql.functions import col, window

data = [
    ("Amit", "2026-10-12 10:00:00"),
    ("Rahul", "2026-10-12 10:02:00"),
    ("Amit", "2026-10-12 10:01:00")
]

df = spark.createDataFrame(
    data,
    ["name", "event_time"]
)

df = df.withColumn(
    "event_time",
    col("event_time").cast("timestamp")
)

display(df)

In [0]:
#Apply Watermark
watermarked_df = (
    df.withWatermark("event_time", "2 minutes")
)

display(watermarked_df)

In [0]:
from pyspark.sql.functions import window, col

stream_data = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 2)
    .load()
)

events = stream_data.select(
    col("timestamp").alias("event_time"),
    col("value")
)

result = (
    events
    .withWatermark("event_time", "2 minutes") #late event handling ke liye threshold set karta hai.
    .groupBy(
        window(col("event_time"), "1 minute") #events ko one-minute windows mein group karta hai
    )
    .count()#har window mein events count karta hai.
)

query = (
    result.writeStream
    .format("console")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/employee_data/lab1/watermark_checkpoint"
    )
    .trigger(availableNow=True)
    .start()
)

In [0]:
query.stop() #

In [0]:
%sql SHOW TABLES IN default;

In [0]:
%sql
SELECT * FROM default.bronze_new_employees;